# Our Simple Training and Evaluation Script

- First, we will load simple python libraries that should come preloaded.

In [ ]:
import os
import torch
import numpy as np

from torch_brain.utils import seed_everything

In [ ]:
seed_everything(0)

base_dir = os.getcwd().split("notebooks")[0]
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using: {device}")

## Loading the Data

In [ ]:
from neural_decode.dataset.transformer_dataloader import get_train_val_loaders

In [ ]:
path_to_neural_dataset = os.path.join(base_dir, "data", "perich_miller_population_2018", "t_20130819_center_out_reaching")
train_dataset, train_loader, val_dataset, val_loader = get_train_val_loaders(path_to_neural_dataset, batch_size=64)

num_units = len(train_dataset.get_unit_ids())
print()
print(f"Batches in Train Dataset: {len(train_loader)}")
print(f"Batches in Val Dataset: {len(val_loader)}")
print(f"Num Units in Session: {num_units}")

## Loading a Model

In [ ]:
from neural_decode.models.transformer import TransformerNeuralDecoder 
from neural_decode.models.hopfield_only import HopfieldOnlyDecoder
from neural_decode.models.transformer_hopfield import TransformerHopfieldDecoder


In [ ]:
tf_model = TransformerNeuralDecoder(
    num_units=num_units,
    bin_size=10e-3,
    sequence_length=1.0,
    dim_output=2,
    dim_hidden=128,
    n_layers=3,
    n_heads=4,
).to(device)

hopfield_only_model = HopfieldOnlyDecoder(
    num_units=num_units,
    bin_size=10e-3,
    sequence_length=1.0,
    dim_output=2,
    dim_hidden=128,
    n_layers=3,
).to(device)

tf_hopfield_model = TransformerHopfieldDecoder(
    num_units=num_units,
    bin_size=10e-3,
    sequence_length=1.0,
    dim_output=2,
    dim_hidden=128,
    n_layers=3,
    n_heads=4,
).to(device)


In [ ]:
print(tf_model)

In [ ]:
print(hopfield_only_model)

In [ ]:
print(tf_hopfield_model)

## Training a Model

In [ ]:
from neural_decode.training.train_model import train

In [ ]:
train_dataset.transform = tf_model.tokenize
val_dataset.transform = tf_model.tokenize

optimizer = torch.optim.AdamW(tf_model.parameters(), lr=1e-3)

In [ ]:
transformer_r2_log_tf, transformer_loss_log_tf, transformer_train_outputs_tf = train(tf_model, optimizer, train_loader, val_loader,
                                                                            num_epochs=3, device = device)

print()

transformer_r2_log_hop, transformer_loss_log_hop, transformer_train_outputs_hop = train(hopfield_only_model, optimizer, train_loader, val_loader,
                                                                            num_epochs=3, device = device)

print()

transformer_r2_log_tf_hop, transformer_loss_log_tf_hop, transformer_train_outputs_tf_hop = train(tf_hopfield_model, optimizer, train_loader, val_loader,
                                                                            num_epochs=3, device = device)

## Evaluating a Model

In [ ]:
from neural_decode.evaluation.graphing_functions import plot_training_curves

In [ ]:
fig_plot = plot_training_curves(transformer_r2_log_tf, "R2 Scores for Transformer Decoder")

In [ ]:
fig_plot = plot_training_curves(transformer_r2_log_hop, "R2 Scores for Hopfield Only Decoder")

In [ ]:
fig_plot = plot_training_curves(transformer_r2_log_tf_hop, "R2 Scores for Transformer + Hopfield Decoder")

## Feature Embedding Analysis

In [ ]:
from neural_decode.post_hoc_analysis.embedding_analysis import compute_umap_embeddings

In [ ]:
embeddings, computations = compute_umap_embeddings(tf_hopfield_model, val_loader, device=device)

In [ ]:
print(f"UMAP Embeddings Shape: {embeddings.shape}")

## Saliency Based Analysis

In [ ]:
from neural_decode.post_hoc_analysis.saliency_based_analysis import compute_and_return_attribution_maps

In [ ]:
sal_maps = compute_and_return_attribution_maps(tf_hopfield_model, val_loader)

In [ ]:
print(f"Saliency Maps Shape: {sal_maps.shape}")
print(f"Min Value in Saliency Maps: {np.min(sal_maps)}")
print(f"Max Value in Saliency Maps: {np.max(sal_maps)}")